In [ ]:
import re
import json
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
tqdm.pandas()

## NER Model

In [ ]:
import spacy
from span_marker import SpanMarkerModel

In [ ]:
# Load the spacy model with the span_marker pipeline component
nlp = spacy.load("en_core_web_sm", exclude=["ner"])
nlp.add_pipe("span_marker", config={"model": "tomaarsen/span-marker-roberta-large-ontonotes5"})

## Google Maps

In [ ]:
# Load environment variables
import os
from dotenv import load_dotenv

load_dotenv()
gmap_api_key =  os.getenv('GMAP_API_KEY')

In [ ]:
import ast
import requests
import googlemaps
from mapbox import Geocoder

In [ ]:
gmap_client_key = gmap_api_key
gmaps = googlemaps.Client(key=gmap_client_key)

In [ ]:
# Google Maps API handler
def callGoogleMapsAPI(location):
    try:
        # If it's in boston, we can do a more specific search
        if ("Boston" in location):
            location = f"{location}, Boston"
            
        # Locations are limited to Massachusetts for now
        geocode_result = gmaps.geocode(f"{location}, Massachussetts", components={"administrative_area_level": "MA", "country": "US"})
        
        if (len(geocode_result) > 0):
            longitude = geocode_result[0]['geometry']['location']['lng']
            latitude = geocode_result[0]['geometry']['location']['lat']
            return longitude, latitude
        else:
            return None
    except Exception as error:
        print(error)
        return None

### Functions to manage caches

In [ ]:
# Load the cache from the file at the start
def load_cache(path):
    try:
        with open(path, 'r') as file:
            cache = json.load(file)
    except FileNotFoundError:
        cache = {}
    return cache

# Save cache to file
def save_cache_to_file(cache, path):
    with open(path, 'w') as file:
        json.dump(cache, file, indent=4)

## Pipeline Entry Point

In [ ]:
sample_data_path = "./sample_data/Articles_Nov_2020_March_2023.csv" # Using this as I don't have the other one
# sample_data_dir = "./sample_data/se_naacp_db.articles_data.csv"

In [ ]:
# Temporary. Use given article data set. Comment out when obtain the other data sate
full_df = pd.read_csv(sample_data_path)

# Format data set to match expected pipeline input
full_df = full_df.rename(columns={"Headline": "hl1", "Body": "body"})

# Make 'tagging' column be the id column
tagging_col = full_df.pop('Tagging')
full_df.insert(0, '_id', tagging_col)

# Drop rows where at least one of the specified columns is empty
columns_to_check = ['_id', 'hl1', 'body'] 
full_df = full_df.dropna(subset=columns_to_check, how='all')

# Drop empty rows too
full_df = full_df[~full_df['body'].apply(lambda x: isinstance(x, float))]
full_df = full_df[~full_df['hl1'].apply(lambda x: isinstance(x, float))]

# Pick a random sample of 10 articles
raw_df = full_df.sample(10)
# raw_df = full_df
len(raw_df)


The ML Model honestly just needs the `id`, `header`, and `body`.

In [ ]:
df = pd.concat([raw_df['_id'], raw_df['hl1'], raw_df['body']], axis=1)

Remove Duplicates (if any)

In [ ]:
df = df.drop_duplicates(subset=['hl1'])

Clean the HTML in the body and header

In [ ]:
# Function to extract the text from the html of the article
func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
df['body'] = df['body'].progress_apply(func_clean_html)
df['hl1'] = df['hl1'].progress_apply(func_clean_html)

Clean the Body and Header with Regex

In [ ]:
# Function to remove extra symbols from the text
func_clean_regex = lambda text: ' '.join([word for word in re.findall(r'[A-Za-z0-9!@#$%^&*().]+', text) if len(word) > 1])
df['body'] = df['body'].progress_apply(func_clean_regex)
df['hl1'] = df['hl1'].progress_apply(func_clean_regex)

In [ ]:
unwanted_entities_path = "./geodata/unwanted_locations.json"  
unwanted_entities = load_cache(unwanted_entities_path)

In [ ]:
df['Explicit_Pass'] = None

### NER Code First Pass

In [ ]:
# Return the first valid facility found, or organization if none are found
def valid_facility(entities, firstPass):
    print(entities)

    if (firstPass): 
        for entity in entities:
            # If it's a valid facility, return it
            if (entity.label_ == "FAC" and entity.text not in unwanted_entities["FAC"]):
                return entity.text
        else:
            return None
    
    else:
        first_org = None

        for entity in entities:
            # If it's a valid facility, return it
            if (entity.label_ == "FAC" and entity.text not in unwanted_entities["FAC"]):
                return entity.text
            
            # If it's a valid organization, save it (but don't return in case there's a facility later on)
            if (first_org == None and entity.label_ == "ORG" and entity.text not in unwanted_entities["ORG"]):
                first_org = entity.text
        else:             
            return first_org # Return regardless of whether it's None or not 


In [ ]:
# Run NER on the body of the article and return first valid facility
def predict_NER_def(text, firstPass=True):
    try:
        if (text == None or text == ""):
            return None
        
        entities = nlp(text).ents
        return valid_facility(entities, firstPass)
        
    except Exception as error:
        return error

In [ ]:
# Run NER on the articles that do not have an explicit location in the title
def explicit_filtering_NER(article):
    try:
        # If the article does not have an explicit location, run NER
        if (article['Explicit_Pass'] != None): 
            print(f"Has location from title: {article['hl1']}")
            return None
        else:
            return predict_NER_def(article['body'])
    except Exception as error:
        print(error)
        return error

## MultiProcessing Attempt

In [ ]:
chunk_size = 100
# Split article text into chunks of specified size
def split_text_into_chunks(text, chunk_size=chunk_size):
    words = text.split()
    chunks = [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]
    return chunks

# Chunk processing
chunk_size = 100
def chunk_processing(text, chunk_size=chunk_size):
    if (text is None or text == ""):
            return None

    # Split text into smaller chunks
    chunks = split_text_into_chunks(text, chunk_size)
    
    # Process each chunk and return if a valid facilty is found
    for chunk in chunks:
        result = predict_NER_def(chunk)
        if result is not None:
             return result
    return None


In [ ]:
def run_pipeline(text):
    try: 
        import pandas as pd
        import spacy
        from span_marker import SpanMarkerModel

        # Load the spacy model with the span_marker pipeline component
        nlp = spacy.load("en_core_web_sm", exclude=["ner"])
        nlp.add_pipe("span_marker", config={"model": "tomaarsen/span-marker-roberta-large-ontonotes5"})
                
        # Run NER on the body of the article and return first valid facility
        if (text == None or text == ""):
            return None
            
        # Return first valid facility found
        entities = nlp(text).ents
        for entity in entities:
            # If it's a valid facility, return it
            if (entity.label_ == "FAC" and entity.text not in unwanted_entities["FAC"]):
                return entity.text
        return None
                
    except Exception as error:
        print(error)
        return error

In [ ]:
import ipyparallel as ipp
import multiprocessing as mp

# Start and connect to an IPyParallel cluster
rc = ipp.Cluster(n=4).start_and_connect_sync() # mp.cpu_count() - 2
dview = rc[:]

# Process the articles parallelly using ipyparallel
def process_articles(data):
    text_list = data['body'].tolist()
    
    print("Collecting results from ipyparallel map...")
    results = dview.map_sync(run_pipeline, text_list)
    return results

data = {
    'body': [
        "The new medical center, Boston General Hospital, has opened its doors to the public this week, offering state-of-the-art medical services to the community.",
        "The conference at Stanford University was a success, bringing together experts from various fields to discuss advancements in artificial intelligence.",
        "Central Park Zoo announced the birth of a rare white tiger, attracting visitors from all over the world to see the new addition.",
        "Microsoft's headquarters in Redmond are known for their innovation and cutting-edge technology development.",
        "The annual tech summit held at Silicon Valley was attended by representatives from Google, Apple, and Facebook.",
        "The renovation of the Los Angeles Public Library has been completed, providing improved facilities for reading and research.",
        "The seminar on climate change at Harvard University was well-received, with prominent scientists presenting their latest research findings.",
        "The opening of the new wing at the Smithsonian Museum has drawn large crowds eager to see the latest exhibits.",
        "Mayo Clinic in Rochester is renowned for its advanced medical treatments and patient care.",
        "The art exhibition at the Louvre Museum in Paris features works from renowned artists across different centuries."
    ]
}

df = pd.DataFrame(data)

# Process the articles using ipyparallel
results = process_articles(df)
print(results)

In [ ]:
import time

start_time = time.time()
df['NER_Pass'] = df.progress_apply(explicit_filtering_NER, axis=1)
total_time = time.time() - start_time
print(f"Total Time: {total_time}")


In [ ]:
df.head(10)

In [ ]:
df['NER_Prediction'] = None

Extract locations from the most specific pass

In [ ]:
# Get the locations from the most specific pass for a given article
def extractLocations(article):
    for key in ['Explicit_Pass', 'NER_Pass', 'NER_Prediction']:
        location = article.get(key)
        if location is not None:
            return location
    return None

In [ ]:
df['Locations'] = df.progress_apply(extractLocations, axis=1)

In [ ]:
df.head(10)

In [ ]:
df

## Get the coordinates

In [ ]:
known_locations_path = "./geodata/known_locations.json"  
known_locations = load_cache(known_locations_path)

In [ ]:
# Get the coordinates of the location
def getCoordinates(location): # Valid labels are FAC for NER_Pass; FAC and ORG for NER_Prediction
    if (location == None or len(location) == 0): return None  
    
    # Only get coordinates if the location is not already known
    if (location in known_locations):
        longitude, latitude = known_locations[location]["coordinates"]
    else:
        # Get coordinates and save to cache
        longitude, latitude = callGoogleMapsAPI(location)
        known_locations[location] = {"coordinates": [longitude, latitude], "tract": None, "county": None}
        save_cache_to_file(known_locations, known_locations_path)

    return [longitude, latitude]

In [ ]:
df['Coordinates'] = df['Locations'].progress_apply(getCoordinates)

In [ ]:
df.head(10)

## Geocode locations

In [ ]:
# Get the census tract of the location
def query_census_api(location, coordinates):
    longitude, latitude = coordinates
    base_url = f'https://geocoding.geo.census.gov/geocoder/geographies/coordinates?'
    survey_ver = f'&benchmark=4&vintage=4&layers=2020 Census Blocks&format=json'
    url = f'{base_url}x={longitude}&y={latitude}{survey_ver}'

    response = requests.get(url)

    # Check if response is valid
    if (response.status_code == 200):
        results = response.json()
        try:
            tract = results['result']['geographies']['2020 Census Blocks'][0]['TRACT']
            county = results['result']['geographies']['2020 Census Blocks'][0]['COUNTY']

            return tract, county
        except IndexError:
            print("Unable to retrieve census geography for: " + location)
        except KeyError:
            print("Location is outside of the United States: " + location)
        except Exception as error:
            print(error)

    return None, None  # Return this if API call failed or no tracts found

In [ ]:
# Get the census tract and county of the location
def geocode(location):
    if (location is None or len(location) == 0): return None, None  

    # Only geocode if it's not known
    Tract = known_locations[location]["tract"]
    County = known_locations[location]["county"]
    if (Tract is None or County is None):
        # Geocode article
        coordinates = known_locations[location]["coordinates"]
        Tract, County = query_census_api(location, coordinates)

        # Save to cache
        known_locations[location]["tract"] = Tract
        known_locations[location]["county"] = County
        save_cache_to_file(known_locations, known_locations_path)
    
    return Tract, County
    

In [ ]:
df[['Tracts', 'County']] = df.progress_apply(lambda row: pd.Series(geocode(row['Locations'])), axis=1)
df

In [ ]:
print(df['Explicit_Pass'].value_counts().sum())
df['Explicit_Pass'].value_counts()

In [ ]:
print(df['NER_Pass'].value_counts().sum())
df['NER_Pass'].value_counts()

In [ ]:
print(df['NER_Prediction'].value_counts().sum())
df['NER_Prediction'].value_counts()

In [ ]:
df

In [ ]:
df.head(10)

In [ ]:
len(df)

In [ ]:
df = df.dropna(subset=["Tracts", "County"]) # Clean those that don't have a Tract or a County

In [ ]:
print(len(df))
df.head(10)

## Topic Modeling

In [ ]:
import os
import tiktoken
import numpy as np
from transformers import pipeline
from sklearn.metrics import adjusted_rand_score
from openai import OpenAI, AsyncOpenAI
from sklearn.metrics.pairwise import cosine_similarity
from tenacity import retry, wait_random_exponential, stop_after_attempt

## OpenAI Client

In [ ]:
# Retry up to 10 times with exponential backoff, starting at 1 second and maxing out at 20 seconds delay
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(10))
def get_embedding(text: str, model="text-embedding-3-small"):
    #print(text)
    try:
        embedding = client.embeddings.create(input=text, model=model).data[0].embedding
        return embedding
    except Exception as e:
        print(f"Failed to retrieve ADA Embedding: {e}. Replacing with replacement value!")
        return [-1.0]
    return 

In [ ]:
client = OpenAI(
    api_key='YOUR_KEY_HERE',
)

## Taxonomy Lists

Content Taxanomy

In [ ]:
# Get the embedding for taxonomy
taxonomy_df = pd.read_csv('./taxonomy_list/Content_Taxonomy.csv', skiprows=5, usecols=range(8))
taxonomy_df.columns = taxonomy_df.iloc[0]
taxonomy_df = taxonomy_df.tail(-1)

tier_1_list = []
tier_2_list = []
tier_3_list = []
tier_4_list = []
for index, row in taxonomy_df.iterrows():
    if not pd.isnull(row['Tier 4']) and row['Tier 4'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_4_label = row['Tier 4']
        tier_4_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label} - {tier_4_label}')
    elif not pd.isnull(row['Tier 3']) and row['Tier 3'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_3_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label}')
    elif not pd.isnull(row['Tier 2']) and row['Tier 2'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_2_list.append(f'{tier_1_label} - {tier_2_label}')
    else:
        tier_1_label = row['Tier 1']
        tier_1_list.append(f'{tier_1_label}')

tier_1_list = list(set(tier_1_list))
tier_2_list = list(set(tier_2_list))
tier_3_list = list(set(tier_3_list))
tier_4_list = list(set(tier_4_list))

tier_1_embedding = [get_embedding(topic) for topic in tier_1_list]
tier_2_embedding = [get_embedding(topic) for topic in tier_2_list]
tier_3_embedding = [get_embedding(topic) for topic in tier_3_list]
tier_4_embedding = [get_embedding(topic) for topic in tier_4_list]

all_topics_list = []
[all_topics_list.append(topic) for topic in tier_1_list]
[all_topics_list.append(topic) for topic in tier_2_list]
[all_topics_list.append(topic) for topic in tier_3_list]
[all_topics_list.append(topic) for topic in tier_4_list]

all_topics_embedding = []
[all_topics_embedding.append(embedding) for embedding in tier_1_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_2_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_3_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_4_embedding]
print(len(all_topics_embedding))

Selected Taxonomy List

In [ ]:
# Get embedding for the 230 topics selected by BERTopic 
selected_taxonomy_df = pd.read_csv('./topics/embedding_similarity_label.csv')
selected_taxonomy_df = selected_taxonomy_df.dropna(subset=['closest_topic'])
selected_topics_list = selected_taxonomy_df['closest_topic'].values.tolist()

selected_topics_embedding = [get_embedding(topic) for topic in selected_topics_list]

Client Taxonomy List

In [ ]:
# Alternative taxonomy: client's list of topics
client_taxonomy_df = pd.read_excel('./topics/Asad_Topics_List.xlsx', names=['label'])
client_taxonomy_df['ada_embedding'] = client_taxonomy_df['label'].map(get_embedding)

## Obtaining Ada Embedding

In [ ]:
def truncate(tokens, length=500):
    """
    Function to get the first 500 elements from a list
    """
    return tokens[:length]

In [ ]:
df['topic_model_body'] = df['body'].apply(lambda x: re.sub(re.compile('<.*?>'), '', x))
df['tokens'] = df['topic_model_body'].apply(lambda x: x.split())
df['tokens'] = df['tokens'].apply(truncate)

In [ ]:
df['ada_embedding'] = df.tokens.apply(lambda x: get_embedding(','.join(map(str,x)), model='text-embedding-3-small'))

## Similarity Matching After Ada Embedding

In [ ]:
# Find most similar taxonomy (out of all toipcs) to news body
closest_topic_list_all = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in all_topics_embedding]

    # Find the index of the topic with the highest similarity
    closest_topic_index = np.argmax(similarities)

    # Retrieve the closest topic embedding
    closest_topic = all_topics_list[closest_topic_index]
    closest_topic_list_all.append(closest_topic)

df['closest_topic_all'] = closest_topic_list_all

In [ ]:
# Find most similar taxonomy (out of 230 selected topics) to news body
closest_topic_list_selected = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in selected_topics_embedding]

    # Find the index of the topic with the highest similarity
    closest_topic_index = np.argmax(similarities)

    # Retrieve the closest topic embedding
    closest_topic = selected_topics_list[closest_topic_index]
    closest_topic_list_selected.append(closest_topic)

df['closest_topic_selected'] = closest_topic_list_selected

In [ ]:
client_topic_embedding_list = client_taxonomy_df['ada_embedding'].to_list()
client_topic_list = client_taxonomy_df['label'].to_list()
similarity_arr = []

closest_topic_list_client = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in client_topic_embedding_list]
    
    if max(similarities) > 0.25:    
        closest_topic_index = np.argmax(similarities) # Find the index of the topic with the highest similarity
        closest_topic = client_topic_list[closest_topic_index] # Retrieve the closest topic embedding
        closest_topic_list_client.append(closest_topic)
    else:
        closest_topic_list_client.append('Other')
    similarity_arr.append(max(similarities))
    
df['closest_topic_client'] = closest_topic_list_client

In [ ]:
df

In [ ]:
df.to_csv("./outputs/gbh_output.csv")

In [ ]:
raw_df

In [ ]:
df

In [ ]:
merged_df = pd.merge(raw_df, df, on='_id', how='inner')

In [ ]:
merged_df

In [ ]:
merged_df.to_csv("./outputs/gbh_output_all_fields.csv")

In [ ]:
merged_df.columns